# Parte 3 – Ritmos cerebrales

PSD de Welch (ventana y solape justificados), potencia absoluta y relativa en delta, theta, alfa y beta, cociente theta/beta. Verificación con modelo AR (orden por AIC/BIC) en algunos sujetos.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

%load_ext autoreload
%autoreload 2

In [2]:
from eeg_nback import datos, filtrado, epocas, espectro

## 1. Potencia en una banda: `potencia_banda`

La PSD dice cómo se reparte la potencia de la señal entre las frecuencias, en µV²/Hz. La potencia de una banda es el **área bajo la PSD** entre sus dos límites, en µV². Como la PSD está calculada en frecuencias discretas, el área se aproxima con la regla del trapecio:

$$P_{banda} = \int_{f_1}^{f_2} S(f)\,df \approx \sum_k \frac{S(f_k) + S(f_{k+1})}{2}\,\Delta f$$

Antes de usarla con el EEG se prueba con señales cuya potencia se conoce de antemano, igual que se hizo con el LMS en la Parte 1. Las señales tienen el mismo formato que las épocas reales: 2 s a 256 Hz. En esta sección el segmento de Welch es de 1 s; el largo definitivo se decide en la sección 2.

### 1.1 Prueba con una senoidal

Una senoidal de amplitud $A$ tiene potencia $A^2/2$. Si tiene 10 Hz, toda esa potencia tiene que aparecer en alfa y casi nada en las otras bandas.

In [3]:
import numpy as np
import pandas as pd
from scipy import integrate
from eeg_nback.config import load_config

cfg = load_config()
fs, bandas = cfg['FS'], cfg['BANDAS']
rng = np.random.default_rng(cfg['SEED'])

A = 20.0                                   # µV
t = np.arange(2 * fs) / fs                 # una época de 2 s
fases = rng.uniform(0, 2 * np.pi, (500, 1))
x = A * np.sin(2 * np.pi * 10 * t + fases) # 500 épocas con fase al azar

f, pxx = espectro.psd_welch(x, fs, nperseg=fs)
pxx = pxx.mean(axis=0)                     # PSD promedio entre épocas

print(f'Resolución: {f[1] - f[0]} Hz.  Potencia esperada en alfa: A²/2 = {A**2 / 2:.1f} µV²')
pd.Series({b: espectro.potencia_banda(f, pxx, r) for b, r in bandas.items()}).round(3)

Resolución: 1.0 Hz.  Potencia esperada en alfa: A²/2 = 200.0 µV²


delta      0.0
theta      0.0
alfa     200.0
beta       0.0
dtype: float64

### 1.2 Prueba con ruido blanco: por qué la máscara incluye los dos bordes

El ruido blanco de varianza $\sigma^2$ tiene una PSD plana de valor $2\sigma^2/f_s$, así que la potencia de una banda es proporcional a su ancho: $P = 2\sigma^2 (f_2 - f_1)/f_s$. Se compara ese valor con dos máscaras posibles:

- `f < f_max`: deja afuera la frecuencia del límite superior, así que el trapecio termina un bin antes.
- `f <= f_max`: incluye los dos bordes (la que usa `potencia_banda`).

In [4]:
sigma = 10.0
ruido = rng.normal(0, sigma, (2000, 2 * fs))
f, pxx = espectro.psd_welch(ruido, fs, nperseg=fs)
pxx = pxx.mean(axis=0)

def con_borde_abierto(f, pxx, banda):
    m = (f >= banda[0]) & (f < banda[1])
    return integrate.trapezoid(pxx[m], f[m])

tabla = pd.DataFrame({
    'teorica': {b: 2 * sigma**2 * (r[1] - r[0]) / fs for b, r in bandas.items()},
    'f < f_max': {b: con_borde_abierto(f, pxx, r) for b, r in bandas.items()},
    'f <= f_max': {b: espectro.potencia_banda(f, pxx, r) for b, r in bandas.items()},
})
tabla['error_abierto_%'] = 100 * (tabla['f < f_max'] / tabla['teorica'] - 1)
tabla['error_cerrado_%'] = 100 * (tabla['f <= f_max'] / tabla['teorica'] - 1)
print(tabla.round(2))

total = espectro.potencia_banda(f, pxx, (1, 30))
print(f"\nPotencia total 1-30 Hz: {total:.3f} µV²")
print(f"Suma de las bandas con f <= f_max: {tabla['f <= f_max'].sum():.3f} µV²")
print(f"Suma de las bandas con f < f_max:  {tabla['f < f_max'].sum():.3f} µV²")

       teorica  f < f_max  f <= f_max  error_abierto_%  error_cerrado_%
delta     2.34       1.48        2.23           -37.03            -4.74
theta     3.12       2.30        3.08           -26.47            -1.45
alfa      3.91       3.10        3.88           -20.53            -0.70
beta     13.28      12.56       13.35            -5.42             0.53

Potencia total 1-30 Hz: 22.542 µV²
Suma de las bandas con f <= f_max: 22.542 µV²
Suma de las bandas con f < f_max:  19.439 µV²


**Qué vemos:**

- **Senoidal:** los 200,0 µV² esperados aparecen enteros en alfa y las otras bandas dan 0,0. La función integra bien y la escala de la PSD es correcta.
- **Con `f < f_max` se pierde potencia:** delta queda 37 % por debajo del valor teórico, theta 26 %, alfa 21 % y beta 5 %. El error es mayor cuanto más angosta es la banda, porque el tramo perdido mide siempre un bin ($\Delta f$ = 1 Hz). Eso distorsionaría la potencia relativa y el cociente theta/beta.
- **Con `f <= f_max`:** las bandas se tocan sin huecos, y su suma (22,542 µV²) coincide con la potencia total entre 1 y 30 Hz. Con la máscara abierta la suma da 19,439 µV².
- **Queda un error de −4,7 % en delta.** No viene de la integración sino de Welch, que le resta la media a cada segmento: con segmentos de 1 s, eso baja la PSD en el bin de 1 Hz al 83 % de su valor real. Con segmentos de 2 s ($\Delta f$ = 0,5 Hz), el error en delta baja a −1,5 %. Afecta sobre todo a delta, y hay que tenerlo en cuenta al elegir el largo del segmento de Welch.

## 2. Largo del segmento de Welch: por qué 2 s

Welch tiene un compromiso: con segmentos largos la resolución en frecuencia es mejor, y con segmentos cortos hay más segmentos para promediar y la estimación varía menos. El límite superior lo pone la época, que dura 2 s: un segmento más largo cruzaría de un ensayo al siguiente y podría incluir una época rechazada. Las opciones razonables son entonces **1 s** (3 segmentos solapados por época, $\Delta f$ = 1 Hz) o **2 s** (la época entera, $\Delta f$ = 0,5 Hz).

Se comparan tres cosas: la fuga entre bandas vecinas, la varianza del estimador y el efecto sobre el EEG real.

### 2.1 Fuga entre bandas

La ventana de Hann ensancha cada componente: su lóbulo principal ocupa $\pm 2\,\Delta f$. Una senoidal de 12 Hz pertenece a alfa, pero con $\Delta f$ = 1 Hz su energía llega hasta 14 Hz y entra en beta.

In [5]:
rng = np.random.default_rng(cfg['SEED'])
t = np.arange(2 * fs) / fs
x = 20 * np.sin(2 * np.pi * 12 * t + rng.uniform(0, 2 * np.pi, (500, 1)))

filas = {}
for seg, nperseg in {'1 s': fs, '2 s': 2 * fs}.items():
    f, pxx = espectro.psd_welch(x, fs, nperseg=nperseg)
    pxx = pxx.mean(axis=0)
    alfa = espectro.potencia_banda(f, pxx, bandas['alfa'])
    beta = espectro.potencia_banda(f, pxx, bandas['beta'])
    filas[seg] = {'alfa': alfa, 'beta': beta, 'fuga_a_beta_%': 100 * beta / (alfa + beta)}
pd.DataFrame(filas).T.round(2)

,alfa,beta,fuga_a_beta_%
1 s,183.33,16.67,8.33
2 s,200.00,0.00,0.00


**Qué vemos:** con segmentos de 1 s, el 8,33 % de la potencia de una senoidal de 12 Hz se cuenta como beta. Con 2 s, la fuga es 0. Como beta es la banda central del trabajo, conviene que no reciba energía de alfa.

### 2.2 Varianza del estimador

Se mide con ruido blanco el coeficiente de variación (desvío / media) de la potencia en dos casos: en una sola época, y promediando 168 épocas, que es lo que tiene cada tipo de bloque en una sesión (8 bloques × 21 épocas).

In [6]:
ruido = rng.normal(0, 10, (20000, 2 * fs))
filas = {}
for seg, nperseg in {'1 s': fs, '2 s': 2 * fs}.items():
    f, pxx = espectro.psd_welch(ruido, fs, nperseg=nperseg)
    for banda in ('theta', 'beta'):
        v = espectro.potencia_banda(f, pxx, bandas[banda])
        grupos = v[:len(v) // 168 * 168].reshape(-1, 168).mean(axis=1)
        filas[(seg, banda)] = {'CV_1_epoca_%': 100 * v.std() / v.mean(),
                               'CV_168_epocas_%': 100 * grupos.std() / grupos.mean()}
pd.DataFrame(filas).T.round(2)

CV_1_epoca_%  CV_168_epocas_%
1 s theta         37.35             2.78
    beta          19.56             1.39
2 s theta         47.40             3.49
    beta          23.60             1.76

**Qué vemos:** el segmento de 2 s es peor por época (en beta, 23,6 % contra 19,6 %), porque cada época aporta un solo segmento en lugar de tres. Ese es su costo. Pero al promediar 168 épocas la diferencia casi desaparece: 1,76 % contra 1,39 % en beta y 3,49 % contra 2,78 % en theta. La PSD final siempre se promedia entre épocas, así que esa es la varianza que importa.

### 2.3 Efecto sobre el EEG real

Se comparan las dos opciones en las épocas de tarea del sujeto **A1**, con los criterios de rechazo de la Parte 2. `usable` son los criterios sobre 1-40 Hz y `usable_beta`, sobre 13-40 Hz. Por simplicidad se usan solo los criterios por época, sin la regla por canal de la Parte 2. La tabla muestra el cociente entre la potencia con segmentos de 1 s y la potencia con segmentos de 2 s: 1 significa que la elección no cambia nada.

In [7]:
CANALES = cfg['CANALES']
UMBRALES = dict(umbral_pp=150.0, umbral_z=3.5, umbral_curtosis=5.0, desvio_min=1.0)
sujeto = datos.listar_sujetos().query("sujeto == 'A1'").iloc[0]

def epocas_validas(sos):
    ep, info, _ = epocas.epocas_sujeto(sujeto.ruta, fs, CANALES, sos)
    tabla = epocas.marcar_artefactos(epocas.tabla_criterios(ep, info, CANALES), **UMBRALES)
    return ep, info, tabla.pivot(index='epoca', columns='canal', values='valida')[CANALES]

ep, info, usable = epocas_validas(filtrado.disenar_pasabanda(fs))
_, _, usable_beta = epocas_validas(filtrado.disenar_pasabanda(fs, 13, 40))

tarea = info.tipo.isin(['1-back', '3-back']).to_numpy()
X = np.moveaxis(ep, 1, 2)                  # épocas × canales × muestras
f1, p1 = espectro.psd_welch(X, fs, nperseg=fs)
f2, p2 = espectro.psd_welch(X, fs, nperseg=2 * fs)

filas = {}
for j, canal in enumerate(CANALES):
    m = usable[canal].to_numpy() & tarea
    filas[canal] = {'epocas': m.sum()} | {
        b: espectro.potencia_banda(f1, p1[m, j].mean(axis=0), r)
           / espectro.potencia_banda(f2, p2[m, j].mean(axis=0), r)
        for b, r in bandas.items()}
pd.DataFrame(filas).T.round(3)

,epocas,delta,theta,alfa,beta
TP9,118.0,0.883,1.024,0.960,0.981
AF7,1167.0,0.920,1.002,1.006,1.004
AF8,1169.0,0.922,1.003,0.996,0.993
TP10,127.0,0.881,1.021,1.008,0.977


**Qué vemos:** en theta, alfa y beta las dos opciones difieren hasta un 4 %, y en los canales frontales, que tienen más de 1100 épocas, en menos del 1 %. La diferencia aparece en delta, donde el segmento de 1 s da entre 8 y 12 % menos, por el sesgo en 1 Hz visto en la sección 1.2.

**Decisión: segmentos de 2 s**, o sea, la época entera. Razones:

1. **Coincide con la unidad de análisis:** cada época es un ensayo completo (Parte 2).
2. **Mejor resolución:** $\Delta f$ = 0,5 Hz en lugar de 1 Hz.
3. **Menos fuga entre bandas:** 0 % contra 8,33 % en la prueba de la sección 2.1.
4. **Menos sesgo en delta:** −1,5 % contra −4,7 % con ruido blanco.

El costo es más varianza por época, que se compensa al promediar. El promedio de Welch no desaparece: en lugar de promediar segmentos dentro de una época, se promedian las épocas válidas de cada condición. Con el EEG de A1, la elección casi no cambia theta, alfa ni beta, así que los resultados no dependen de esta decisión.

## 3. Todas las bandas: `potencias`

`potencias` devuelve la potencia absoluta y relativa de cada banda y el cociente theta/beta. Como las cuatro bandas cubren exactamente de 1 a 30 Hz, las potencias relativas tienen que sumar 1.

### 3.1 Verificación

In [8]:
pot = espectro.potencias(f2, p2, bandas)
suma = sum(pot[f'{b}_rel'] for b in bandas)
print('Forma de cada resultado (épocas × canales):', pot['beta_abs'].shape)
print(f'Suma de las potencias relativas: mínimo {suma.min():.6f}, máximo {suma.max():.6f}')

Forma de cada resultado (épocas × canales): (1728, 4)
Suma de las potencias relativas: mínimo 1.000000, máximo 1.000000


**Qué vemos:** las potencias relativas suman 1,000000 en todas las épocas y canales, así que las bandas cubren el rango de 1 a 30 Hz sin huecos.

### 3.2 Qué épocas sirven para cada medida

La Parte 2 definió dos niveles de rechazo, porque las deflexiones lentas de TP9 y TP10 no afectan a beta. Pero la potencia relativa y el cociente theta/beta incluyen las frecuencias bajas. Se compara beta en dos grupos de épocas de tarea:

- **ambas:** pasan los dos niveles de rechazo.
- **solo beta:** pasan `usable_beta` pero no `usable`, es decir, tienen una deflexión lenta.

In [9]:
filas = {}
for j, canal in enumerate(CANALES):
    a = usable[canal].to_numpy() & usable_beta[canal].to_numpy() & tarea
    b = usable_beta[canal].to_numpy() & ~usable[canal].to_numpy() & tarea
    filas[canal] = {
        'n_ambas': a.sum(), 'n_solo_beta': b.sum(),
        'beta_rel_ambas': np.median(pot['beta_rel'][a, j]),
        'beta_rel_solo_beta': np.median(pot['beta_rel'][b, j]),
        'beta_abs_ambas': np.median(pot['beta_abs'][a, j]),
        'beta_abs_solo_beta': np.median(pot['beta_abs'][b, j]),
    }
pd.DataFrame(filas).T.round(3)

,n_ambas,n_solo_beta,beta_rel_ambas,beta_rel_solo_beta,beta_abs_ambas,beta_abs_solo_beta
TP9,118.0,1109.0,0.411,0.028,31.566,43.916
AF7,1164.0,47.0,0.709,0.089,30.855,31.981
AF8,1168.0,53.0,0.716,0.377,50.616,29.925
TP10,126.0,1100.0,0.442,0.028,32.142,43.604


**Qué vemos:**

- **La potencia relativa no sirve en las épocas "solo beta".** En TP9, la mediana de beta relativa cae de 0,411 a 0,028, unas 15 veces, y en TP10 de 0,442 a 0,028. La deflexión lenta no agrega potencia en beta: agranda el denominador. Lo mismo vale para el cociente theta/beta.
- **Consecuencia para el análisis:** con `usable` solo quedan 118 épocas en TP9 y 126 en TP10, contra 1227 y 1226 con `usable_beta`. Para usar los canales temporales, **la medida principal de beta tiene que ser la potencia absoluta** (`beta_abs`). La potencia relativa y el cociente theta/beta se calculan solo en épocas `usable`, en la práctica en los canales frontales.
- **Queda una duda para revisar.** En A1, la potencia beta absoluta es cerca de 39 % mayor en las épocas "solo beta" de TP9 (43,9 contra 31,6 µV²) y 36 % mayor en TP10. En B1, la Parte 2 midió que las deflexiones no cambian beta (×0,9 a ×1,0). Los grupos tienen épocas distintas y el de "ambas" es chico, así que este dato no alcanza para concluir: hay que comprobarlo en los 20 sujetos antes de usar `beta_abs` en TP9 y TP10.